# GMGI v2 Colab-first setup
Run these cells first in Colab. They mount Drive, install runtime dependencies, start Ollama, and configure the app to use the Diffusers creative backend plus persistent self-improvement state.


In [ ]:
# Colab Drive mount and persistent paths
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/gmgi')
except Exception:
    DRIVE_ROOT = Path('experiments/colab_local')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['GMGI_EXPERIENCE_STORE'] = str(DRIVE_ROOT / 'experience_store.jsonl')
os.environ['GMGI_BANDIT_STATE'] = str(DRIVE_ROOT / 'bandit_state.json')
os.environ['GMGI_PROMPT_VERSIONS'] = str(DRIVE_ROOT / 'prompt_versions')
os.environ.setdefault('GMGI_CREATIVE_BACKEND', 'diffusers')
os.environ.setdefault('GMGI_SDXL_MODEL', 'stabilityai/sdxl-turbo')
os.environ.setdefault('GMGI_DISABLE_CLIP_CRITIC', '0')
print('GMGI Drive root:', DRIVE_ROOT)


In [ ]:
# Python dependencies for Ollama-backed agents and Diffusers generation
%pip install -q instructor smolagents diffusers transformers accelerate safetensors ollama open-clip-torch pycloudflared


In [ ]:
# Install, start, and seed Ollama for local structured agents
!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve > /tmp/ollama.log 2>&1 &
!sleep 5
!ollama pull llama3.2:latest

import os
os.environ.setdefault('GMGI_OLLAMA_MODEL', 'llama3.2:latest')
os.environ.setdefault('GMGI_OLLAMA_HOST', 'http://127.0.0.1:11434')
os.environ.setdefault('GMGI_OLLAMA_BASE_URL', 'http://127.0.0.1:11434/v1')


In [ ]:
# Optional: instantiate the service with persistent state paths
import os
from src.api.service import AgencyConsoleService

service = AgencyConsoleService(
    experience_store_path=os.environ['GMGI_EXPERIENCE_STORE'],
    bandit_state_path=os.environ['GMGI_BANDIT_STATE'],
    prompt_versions_dir=os.environ['GMGI_PROMPT_VERSIONS'],
)
print('Experience episodes loaded:', len(service.experience_store.episodes))


In [ ]:
# Manual self-improvement trigger after enough completed sessions
from pathlib import Path
from src.agents.prompt_optimizer import PromptOptimizerAgent
from src.agents.llm import create_llm

optimizer = PromptOptimizerAgent(
    llm=create_llm(),
    store=service.experience_store,
    config_dir=Path('src/agents/configs'),
)
print('Updated prompts:', optimizer.run())


In [ ]:
!git clone https://github.com/kushal-bhargav/learning.git

## 0. Install and start Ollama for local agents

Run these first in Colab. They install Ollama, start the local server, and pull the model used by Instructor, smolagents, and plain Ollama chat.

In [ ]:
!apt-get update -qq

!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

!ollama serve > /tmp/ollama.log 2>&1 &

!sleep 8

!ollama pull llama3.2:latest

!ollama list

In [ ]:
!python -m pip install -q datasets instructor openai "smolagents[litellm]" diffusers transformers accelerate safetensors ollama pyngrok tqdm

In [ ]:
import os

from getpass import getpass


os.environ["GMGI_OLLAMA_MODEL"] = os.getenv("GMGI_OLLAMA_MODEL", "llama3.2:latest")

os.environ["GMGI_OLLAMA_HOST"] = os.getenv("GMGI_OLLAMA_HOST", "http://localhost:11434")

os.environ["GMGI_OLLAMA_BASE_URL"] = os.getenv("GMGI_OLLAMA_BASE_URL", "http://localhost:11434/v1")

os.environ["GMGI_FORCE_OLLAMA_AGENTS"] = os.getenv("GMGI_FORCE_OLLAMA_AGENTS", "1")

os.environ["GMGI_ALLOW_AGENT_FALLBACK"] = os.getenv("GMGI_ALLOW_AGENT_FALLBACK", "0")

os.environ["GMGI_USE_DEMO_AGENT_RESPONSES"] = os.getenv("GMGI_USE_DEMO_AGENT_RESPONSES", "0")

os.environ["GMGI_INTENT_METHOD"] = os.getenv("GMGI_INTENT_METHOD", "llm_structured")

os.environ["GMGI_PLANNING_METHOD"] = os.getenv("GMGI_PLANNING_METHOD", "llm_structured")

os.environ["GMGI_REQUIRE_FULL_GAN_CHECKPOINT"] = os.getenv("GMGI_REQUIRE_FULL_GAN_CHECKPOINT", "1")

os.environ["GMGI_OLLAMA_TIMEOUT_SECONDS"] = os.getenv("GMGI_OLLAMA_TIMEOUT_SECONDS", "180")


# Public tunnel mode. "cloudflare" needs no token; "ngrok" requires NGROK_AUTHTOKEN.

PUBLIC_TUNNEL = os.getenv("PUBLIC_TUNNEL", "cloudflare").lower()

if PUBLIC_TUNNEL == "ngrok" and not os.getenv("NGROK_AUTHTOKEN"):

    token = getpass("Paste NGROK_AUTHTOKEN: ").strip()

    if token:

        os.environ["NGROK_AUTHTOKEN"] = token

# GMGI Gift Creator complete runner


This notebook runs the full Gift Creator stack from a notebook runtime: Python backend, React/Vite frontend, optional GAN data prep, optional full training, optional metrics, and a live Agency Console.


Default path: install dependencies, require a real Ollama-backed agent path, require a full GAN checkpoint, start FastAPI on port 8000, start Vite on port 5173, and open the UI through a public tunnel. Heavy GPU cells are marked optional. Smoke checkpoints are not accepted in real mode.

## 1. Put the notebook in the repository root


In Colab, upload/clone the repo, then set `PROJECT_DIR` to that folder. In Kaggle, set it to the dataset or working copy path.

In [ ]:
PROJECT_DIR = "/content/learning"  # Change this if your repo lives somewhere else.

%cd {PROJECT_DIR}

!pwd

!ls -la | head

## 2. Check runtime


Use a GPU runtime for real GAN training and metric runs. The app itself can run on CPU, but image generation will be slower.

In [ ]:
!python --version

!nvidia-smi || true

!node --version

!npm --version

## 3. Install Python and frontend dependencies


Colab usually already has PyTorch. This installs the project in editable mode and then installs frontend packages.

In [ ]:
!python -m pip install -q --upgrade pip

!python -m pip install -q -e ".[dev]" pyngrok datasets instructor openai "smolagents[litellm]" diffusers transformers accelerate safetensors ollama tqdm

In [ ]:
!npm --prefix frontend install

## 4. Optional: regenerate GAN data prep


Run this only when you want to rebuild `data/gan` from the configured public-domain sources. It may download images and compute embeddings.

In [ ]:
# Optional heavy/data-network step.

!python -m src.gan.data_pipeline --config src/gan/configs/data_pipeline.json

## 5. Check or train a real GAN checkpoint


The backend rejects smoke/pilot checkpoints when `GMGI_REQUIRE_FULL_GAN_CHECKPOINT=1`. If no checkpoint is present, the next cell starts full training with `src/gan/configs/train.json`. This is the real 256px training path and can take a long time on Colab/Kaggle GPU.

In [ ]:
from pathlib import Path

checkpoints = sorted(Path("experiments").glob("run-*/checkpoint-*.pt"))

print("Latest checkpoint:", checkpoints[-1] if checkpoints else "NONE FOUND")

In [ ]:
%%bash
set -e
CONFIG="${GMGI_TRAIN_CONFIG:-src/gan/configs/train.json}"
echo "Runtime GPU visibility:"
if command -v nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv,noheader || true
else
  echo "nvidia-smi not found. Colab/Kaggle may be running on CPU."
fi
python - <<'PY'
import json
import torch
print(json.dumps({
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
PY
if ls experiments/run-*/checkpoint-*.pt >/dev/null 2>&1; then
  echo "Checkpoint exists. The backend will reject smoke/pilot checkpoints when GMGI_REQUIRE_FULL_GAN_CHECKPOINT=1."
  ls -1 experiments/run-*/checkpoint-*.pt | tail -1
else
  if [ ! -f "data/gan/metadata.jsonl" ] || [ ! -f "data/gan/clip_embeddings.npy" ]; then
    echo "GAN data files are missing. Run the data pipeline cell first:"
    echo "python -m src.gan.data_pipeline --config src/gan/configs/data_pipeline.json"
    exit 1
  fi
  echo "No checkpoint found. Starting real GAN training with ${CONFIG}."
  echo "This is not smoke training. It may run for hours depending on GPU/runtime limits."
  python -m src.gan.train --config "${CONFIG}"
fi

## 6. Optional: run inference and evaluation scripts


These regenerate the agency interpolation strip and GAN metric table from the latest checkpoint.

In [ ]:
# Optional metric/artifact regeneration.

!python scripts/gan_interpolation_strip.py

!python eval/gan_metrics.py --metric-samples 16 --lpips-conditions 8 --diversity-samples 8

## 7. Quick backend/API smoke test


This verifies imports and endpoint behavior before starting long-running servers.

In [ ]:
!GMGI_USE_DEMO_AGENT_RESPONSES=1 GMGI_ALLOW_AGENT_FALLBACK=1 python -m pytest tests/test_api_console.py -q


## 8. Start the FastAPI backend


The backend listens on `0.0.0.0:8000`. `GMGI_CORS_ORIGIN_REGEX=.*` is set for notebook proxy URLs; keep this open only for notebook demos, not production.

In [ ]:
import os

import signal

import subprocess

import sys

import time


PROCESSES = globals().setdefault("GMGI_PROCESSES", {})


def stop_process(name):

    proc = PROCESSES.get(name)

    if proc and proc.poll() is None:

        proc.terminate()

        try:

            proc.wait(timeout=5)

        except subprocess.TimeoutExpired:

            proc.kill()


stop_process("backend")

backend_env = os.environ.copy()

backend_env["GMGI_CORS_ORIGIN_REGEX"] = ".*"
backend_env["GMGI_FORCE_OLLAMA_AGENTS"] = "1"
backend_env["GMGI_ALLOW_AGENT_FALLBACK"] = os.environ.get("GMGI_ALLOW_AGENT_FALLBACK", "0")
backend_env["GMGI_USE_DEMO_AGENT_RESPONSES"] = os.environ.get("GMGI_USE_DEMO_AGENT_RESPONSES", "0")
backend_env["GMGI_INTENT_METHOD"] = os.environ.get("GMGI_INTENT_METHOD", "llm_structured")
backend_env["GMGI_PLANNING_METHOD"] = os.environ.get("GMGI_PLANNING_METHOD", "llm_structured")
backend_env["GMGI_REQUIRE_FULL_GAN_CHECKPOINT"] = os.environ.get("GMGI_REQUIRE_FULL_GAN_CHECKPOINT", "1")
backend_env["GMGI_OLLAMA_TIMEOUT_SECONDS"] = os.environ.get("GMGI_OLLAMA_TIMEOUT_SECONDS", "180")

backend_env.setdefault("GMGI_OLLAMA_MODEL", os.environ.get("GMGI_OLLAMA_MODEL", "llama3.2:latest"))

backend_env.setdefault("GMGI_OLLAMA_HOST", os.environ.get("GMGI_OLLAMA_HOST", "http://localhost:11434"))

backend_env.setdefault("GMGI_OLLAMA_BASE_URL", os.environ.get("GMGI_OLLAMA_BASE_URL", "http://localhost:11434/v1"))

backend_env.setdefault("OLLAMA_MODEL", backend_env.get("GMGI_OLLAMA_MODEL", "llama3.2:latest"))
backend_env.setdefault("OLLAMA_HOST", backend_env.get("GMGI_OLLAMA_HOST", "http://localhost:11434"))


backend = subprocess.Popen(

    [sys.executable, "-m", "uvicorn", "src.api.app:app", "--host", "0.0.0.0", "--port", "8000"],

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    env=backend_env,

)

PROCESSES["backend"] = backend

time.sleep(5)

print("Backend PID:", backend.pid, "alive:", backend.poll() is None)

In [ ]:
import json
import time
from urllib.request import urlopen

def get_json(path: str, timeout_seconds: int = 60):
    deadline = time.time() + timeout_seconds
    last_error = None
    url = f"http://127.0.0.1:8000{path}"
    while time.time() < deadline:
        try:
            with urlopen(url, timeout=10) as response:
                raw = response.read().decode("utf-8")
            return json.loads(raw)
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{path} did not return JSON: {last_error}")

print(json.dumps(get_json("/health"), indent=2))
print(json.dumps(get_json("/personas"), indent=2)[:4000])


## 9. Prepare Public URL Mode

The frontend must start before Cloudflare/ngrok creates the public URL. This cell only sets tunnel mode and same-origin API proxy settings.

In [ ]:
import os

PUBLIC_TUNNEL = os.getenv("PUBLIC_TUNNEL", "cloudflare").lower()
BACKEND_URL = ""  # Use Vite same-origin proxy: /personas, /sessions, /artifacts, /health.
FRONTEND_URL = None  # Created later, after Vite is running.

print("Public tunnel mode:", PUBLIC_TUNNEL)
print("Frontend API base:", BACKEND_URL or "same-origin Vite proxy")
print("Run the frontend start cell next, then run the public URL cell after it.")

## 10. Start the React/Vite Agency Console


The frontend is started with `VITE_API_BASE` pointing at the backend URL above, so browser requests reach the notebook VM instead of your local laptop.

In [ ]:
stop_process("frontend")
frontend_env = os.environ.copy()
frontend_env["VITE_API_BASE"] = BACKEND_URL
frontend = subprocess.Popen(
    ["npm", "--prefix", "frontend", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=frontend_env,
)
PROCESSES["frontend"] = frontend
time.sleep(8)
print("Frontend PID:", frontend.pid, "alive:", frontend.poll() is None)
print("Frontend API base:", BACKEND_URL or "same-origin Vite proxy")
print("Public URL is not created in this cell. Run the next public URL cell after this one.")

## 11. Create And Open Public Frontend URL

Run this only after the frontend PID cell says `alive: True`. The cell waits until the public URL responds before printing it.

In [ ]:
import re
import subprocess
import time
from urllib.request import Request, urlopen

def wait_for_public_url(url: str, timeout_seconds: int = 120) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            req = Request(url, headers={"User-Agent": "GMGI-Colab-Healthcheck"})
            with urlopen(req, timeout=10) as response:
                if response.status < 500:
                    return True
        except Exception:
            time.sleep(3)
    return False

def drain_process_log(name: str, lines: int = 80) -> None:
    proc = PROCESSES.get(name)
    if not proc or proc.stdout is None:
        print(f"No process/log for {name}")
        return
    collected = []
    while True:
        line = proc.stdout.readline()
        if not line:
            break
        collected.append(line.rstrip())
        if len(collected) >= lines:
            break
    print("\n".join(collected[-lines:]) or f"No new {name} log lines.")

if "frontend" not in globals() or frontend.poll() is not None:
    drain_process_log("frontend")
    raise RuntimeError("Frontend process is not running. Re-run the frontend start cell first.")

if PUBLIC_TUNNEL == "ngrok":
    from pyngrok import ngrok
    token = os.getenv("NGROK_AUTHTOKEN", "").strip()
    if not token:
        raise RuntimeError("Set NGROK_AUTHTOKEN or switch PUBLIC_TUNNEL to cloudflare.")
    ngrok.set_auth_token(token)
    ngrok.kill()
    FRONTEND_URL = ngrok.connect(5173, "http").public_url.rstrip("/")
else:
    subprocess.run(["wget", "-q", "-O", "/tmp/cloudflared.deb", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
    subprocess.run(["dpkg", "-i", "/tmp/cloudflared.deb"], check=True)
    old = PROCESSES.get("cloudflared")
    if old and old.poll() is None:
        old.terminate()
    cloudflared = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5173", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    PROCESSES["cloudflared"] = cloudflared
    FRONTEND_URL = None
    log_lines = []
    for _ in range(90):
        line = cloudflared.stdout.readline() if cloudflared.stdout else ""
        if line:
            log_lines.append(line.rstrip())
        match = re.search(r"https://[-a-zA-Z0-9.]+trycloudflare.com", line)
        if match:
            FRONTEND_URL = match.group(0).rstrip("/")
            break
        time.sleep(1)
    if not FRONTEND_URL:
        print("Cloudflared logs:")
        print("\n".join(log_lines[-80:]))
        raise RuntimeError("Cloudflare tunnel did not print a public URL.")

print("Waiting for public DNS/HTTP readiness...")
if not wait_for_public_url(FRONTEND_URL):
    print("URL was created but did not become reachable yet. Wait 30 seconds and run this cell again.")
else:
    print("Public URL is reachable.")
print("Open this public URL in a new tab:", FRONTEND_URL)
print("The backend is not separately public; Vite proxies API routes from the same public frontend URL.")

## 12. End-to-end API request example


This creates one backend session from the notebook, useful when debugging before touching the UI.

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/sessions -H "Content-Type: application/json" -d '{"persona_id":"long-distance-partners","agency_slider":0.5,"seed":2026}' | python -m json.tool | head -120

## 13. Server logs and shutdown


Use the log cell when something fails. Run shutdown before restarting ports or ending the notebook.

In [ ]:
def show_process_log(name, lines=120):

    proc = PROCESSES.get(name)

    if not proc or proc.stdout is None:

        print(f"No process/log for {name}")

        return

    collected = []

    while True:

        line = proc.stdout.readline()

        if not line:

            break

        collected.append(line.rstrip())

        if len(collected) >= lines:

            break

    print("\n".join(collected[-lines:]) or f"No new {name} log lines.")


show_process_log("backend")

show_process_log("frontend")

In [ ]:
# # Run when finished or before restarting the app.
# stop_process("cloudflared")
# stop_process("frontend")
# stop_process("backend")
# print("Stopped cloudflared/backend/frontend processes.")